In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

zaryabahmadkhan_2d_slicing_of_imagetbad_dataset_path = kagglehub.dataset_download('zaryabahmadkhan/2d-slicing-of-imagetbad-dataset')
miniredtrout_configdataset_path = kagglehub.dataset_download('miniredtrout/configdataset')

print('Data source import complete.')


Импорт библиотек

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
import numpy as np
import pandas as pd
import os
import scipy.io
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
import shutil

Путь датасета с kaggle

In [ ]:
path = '/kaggle/input/2d-slicing-of-imagetbad-dataset'

In [ ]:
!pip install hydra-core omegaconf
import hydra
from omegaconf import DictConfig, OmegaConf
import os

In [ ]:
!pip install clearml

In [ ]:
%env CLEARML_WEB_HOST=https://app.clear.ml/
%env CLEARML_API_HOST=https://api.clear.ml
%env CLEARML_FILES_HOST=https://files.clear.ml
%env CLEARML_API_ACCESS_KEY=XGMS61QRIXH7G4URU0X7VGTDAIQUVE
%env CLEARML_API_SECRET_KEY=3mpOB_Br37BLBntBoUvtxmfbBfmkbkdfRUYGje8GfL1gwe_ufJd1w4UzuQWOPaAtTWE

env: CLEARML_WEB_HOST=https://app.clear.ml/
env: CLEARML_API_HOST=https://api.clear.ml
env: CLEARML_FILES_HOST=https://files.clear.ml
env: CLEARML_API_ACCESS_KEY=XGMS61QRIXH7G4URU0X7VGTDAIQUVE
env: CLEARML_API_SECRET_KEY=3mpOB_Br37BLBntBoUvtxmfbBfmkbkdfRUYGje8GfL1gwe_ufJd1w4UzuQWOPaAtTWE


In [ ]:
from clearml import Task, Logger

In [ ]:
config_dir = '/kaggle/input/configdataset/config'

def load_hydra_config():
    config_path = "/kaggle/input/configdataset/config"
    config_name = "config"

    with hydra.initialize_config_dir(config_dir=config_path, version_base="1.2"):
        cfg = hydra.compose(config_name=config_name)
c
    cfg.training.epochs = 20
    OmegaConf.set_struct(cfg, False)
    return cfg


Разделение на train test выборку по пациентам

In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split
def create_split():
    im_path = path + '/2D dataset/images'
    lb_path = path + '/2D dataset/labels'
    output_dir = '/content/'

    pairs = []
    for file in os.listdir(im_path):
        if file.endswith('_image.png'):
            label_file = file.replace('_image.png', '_label.png')
            if os.path.exists(os.path.join(lb_path, label_file)):
                pairs.append((file, label_file))
    train_pairs = pairs

    for split, pairs_list in [('train', train_pairs)]:
        img_dir = os.path.join(output_dir, split, 'images')
        lbl_dir = os.path.join(output_dir, split, 'labels')
        os.makedirs(img_dir, exist_ok=True)
        os.makedirs(lbl_dir, exist_ok=True)

        for img_file, lbl_file in pairs_list:
            shutil.copy2(os.path.join(im_path, img_file), os.path.join(img_dir, img_file))
            shutil.copy2(os.path.join(lb_path, lbl_file), os.path.join(lbl_dir, lbl_file))

    return {
        'train_images': os.path.join(output_dir, 'train', 'images'),
        'train_labels': os.path.join(output_dir, 'train', 'labels')
    }

Датасет

In [ ]:
class SegmentationDataset(Dataset):
    def __init__(self, images_dir, labels_dir, transform=None):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.transform = transform

        self.image_files = sorted([f for f in os.listdir(images_dir) if f.endswith('.png')])
        self.label_files = sorted([f for f in os.listdir(labels_dir) if f.endswith('.png')])
    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image_path = os.path.join(self.images_dir, self.image_files[idx])
        label_path = os.path.join(self.labels_dir, self.label_files[idx])
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        label = cv2.imread(label_path, cv2.IMREAD_GRAYSCALE)

        if self.transform:
            transformed = self.transform(image=image, mask=label)
            image = transformed['image']
            label = transformed['mask']
        label = (label > 0).long()

        return image, label

Модель Unet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class UNet(nn.Module):
  def __init__(self,in_channel,out_channel):
    super(self).__init__()
    self.enc1 = self._contrast(in_channel, 64)
    self.pool1 = nn.MaxPool2d(2)
    self.enc2 = self._contrast(64,128)
    self.pool2 = nn.MaxPool2d(2)
    self.enc3 = self._contrast(128,256)
    self.pool3 = nn.MaxPool2d(2)
    self.bottleneck = nn.Sequential(
        nn.Conv2d(256,512,3,padding=1),
        nn.ReLU(),
        nn.BatchNorm2d(512),
        nn.Conv2d(512,512,3,padding=1),
        nn.ReLU(),
        nn.BatchNorm2d(512),
        nn.ConvTranspose2d(512,256,2,stride=2)
    )
    self.dec3 = self._exp(512,256,128)
    self.dec2 = self._exp(256,128,64)
    self.fc = self._final(128,64,out_channel)
    self._init_weights()
  def _contrast(self,in_channel,out_channel):
    return nn.Sequential(
        nn.Conv2d(in_channel,out_channel,3,padding=1).
        nn.ReLU(),
        nn.BatchNorm2d(out_channel),
        nn.Conv2d(out_channel,out_channel,3,padding=1),
        nn.ReLU(),
        nn.BatchNorm2d(out_channel)
    )
  def _exp(self,in_channel,hidden_channel,out_channel):
    return nn.Sequential(
        nn.Conv2d(in_channel,hidden_channel,3,padding=1),
        nn.ReLU(),
        nn.BatchNorm2d(hidden_channel),
        nn.Conv2d(hidden_channel,hidden_chanel,3,padding=1),
        nn.ReLU(),
        nn.BatchNorm2d(hidden_channel),
        nn.ConvTranspose2d(hidden_channel,out_channel,2,stride=2)
    )
  def _final(self,in_channel,hidden_channel,out_channel):
    return nn.Sequential(
        nn.Conv2d(in_channel,hidden_channel,3,padding=1),
        nn.ReLU(),
        nn.BatchNorm2d(hidden_channel),
        nn.Conv2d(hidden_channel,hidden_channel,3,padding=1),
        nn.ReLU(),
        nn.BatchNorm2d(hidden_channel),
        nn.Conv2d(hidden_channel,out_channel,3,padding=1)
    )
  def _init_weights(self):
    for m in self.modules():
      if isinstance(m,nn.Conv2d):
        nn.init.kaiming_normal_(m.weight,mode='fan_in',nonlinearity='relu')
        if m.bias is not None:
          nn.init_constant_(m.bias,0)
      elif isinstance(m,nn.BatchNorm2d):
        nn.init.constant_(m.weight,1)
        nn.init.constant_(m.bias,0)
      elif isinstance(m,nn.ConvTranspose2d):
        nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
        if m.bias is not None:
          nn.init_constant_(m.bias,0)
  def _concat(self,up,bypass):
    a,b,hu,wu = up.shape
    a,b,hb,wb = bypass.shape
    ch = (hb - hu)//2
    cw = (wb-wu)//2
    crop = bypass[:,:,ch:ch+hu,cw:cw+wu]
    return torch.cat([up,crop],dim=1)
  def forward(self,x):
    e1 = self.enc1(x)
    p1 = self.pool1(e1)
    e2 = self.enc2(p1)
    p2 = self.pool2(e2)
    e3 = self.enc3(p2)
    p3 = self.pool3(e3)
    b = self.bottleneck(p3)
    d3 = self._concat(b,e3)
    d3 = self.dec3(d3)
    d2 = self._concat(d3,e2)
    d2 = self.dec2(d2)
    d1 = self._concat(d2,e1)
    out = self.fc(d1)
    return out

Аугментация и предподготовка

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

def transformers():
    cfg = load_hydra_config()
    train_transform = A.Compose([
        A.Resize(height=cfg.transforms.image_size, width=cfg.transforms.image_size),

        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.RandomRotate90(p=0.5),
        A.Affine(
            translate_percent={'x': (-0.05, 0.05), 'y': (-0.05, 0.05)},
            scale=(0.9, 1.1),
            rotate=(-10, 10),
            p=0.5
        ),
        A.RandomBrightnessContrast(
            brightness_limit=0.1,
            contrast_limit=0.1,
            p=0.3
        ),
        A.GaussNoise(p=0.2),

        A.Normalize(mean=[0.5], std=[0.5]),
        ToTensorV2(),
    ])

    val_transform = A.Compose([
        A.Resize(height=cfg.transforms.image_size, width=cfg.transforms.image_size),
        A.Normalize(mean=[0.5], std=[0.5]),
        ToTensorV2(),
    ])

    return train_transform, val_transform

In [ ]:
!pip install torchmetrics
from torchmetrics.segmentation import DiceScore
from torchmetrics import JaccardIndex
from torchmetrics import Precision, Recall


In [ ]:
cfg = load_hydra_config()

def settings(cfg):
    task = Task.init(
        project_name='IMAGETBADAortaSegmentation',
        task_name=f'UNet_20epochs',
        auto_connect_frameworks={'pytorch': True, 'hydra': True}
    )
    cfg_dict = OmegaConf.to_container(cfg,resolve=True)
    task.connect_configuration(cfg_dict)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    paths = create_split()
    train_transform, val_transform = transformers()
    test_dataset = SegmentationDataset(
        images_dir=paths['train_images'],
        labels_dir=paths['train_labels'],
        transform=val_transform
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=cfg.training.batch_size,
        shuffle=False,
        num_workers=cfg.data.num_workers
    )
    model = UNet(cfg.model.in_channel, cfg.model.out_channel).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.training.learning_rate,
        weight_decay=cfg.training.weight_decay
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min')

    metrics = {
        'dice': DiceScore(num_classes=2, average='macro', input_format='index').to(device),
        'iou': JaccardIndex(num_classes=2, task='multiclass', average='macro').to(device),
        'precision': Precision(num_classes=2, task='multiclass', average='macro').to(device),
        'recall': Recall(num_classes=2, task='multiclass', average='macro').to(device)
    }

    return {
        'device': device,
        'model': model,
        'test_loader': test_loader,
        'criterion': criterion,
        'optimizer': optimizer,
        'scheduler': scheduler,
        'metrics': metrics,
        'task': task
    }

In [ ]:
def train_epoch(model,train_loader,criterion,optimizer,device):
  model.train()
  epoch_train_loss = 0
  train_pbar = tqdm(train_loader,desc='Train')
  for idx, (images, labels) in enumerate(train_pbar):
    images, labels = images.to(device), labels.to(device)
    labels = labels.long()
    optimizer.zero_grad()
    output = model(images)
    loss = criterion(output, labels)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    epoch_train_loss += loss.item()

    train_pbar.set_postfix({
        'Loss': f'{loss.item():.4f}',
        'Avg Loss': f'{epoch_train_loss/(idx+1):.4f}'
    })

  train_pbar.close()
  avg_train_loss = epoch_train_loss / len(train_loader)
  return avg_train_loss

In [ ]:
def validate_epoch(model, test_loader, metrics, device):
    model.eval()

    for metric in metrics.values():
        metric.reset()

    val_pbar = tqdm(test_loader, desc='[Val]')

    with torch.no_grad():
        for idx, (images, masks) in enumerate(val_pbar):
            images, masks = images.to(device), masks.to(device)
            masks = masks.long()

            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)

            for metric in metrics.values():
                metric.update(preds, masks)

    val_pbar.close()

    val_metrics = {}
    for name, metric in metrics.items():
        val_metrics[name] = metric.compute().item()

    return val_metrics

In [ ]:
def early_stopping(cur_metric, best_metric, patience_count, patience, min_d, epoch, model):
    better = False
    if cur_metric < best_metric - min_d:
      better = True
    else:
      if cur_metric > best_metric + min_d:
            better = True
    if better:
        best_metric = cur_metric
        patience_count = 0
        best_epoch = epoch
        best_model_weights = model.state_dict().copy()
    else:
        patience_count += 1
        best_model_weights = None
    should_stop = patience_count >= patience
    return should_stop, best_metric, patience_count, best_model_weights

In [ ]:
def checkpoint(model, optimizer, scheduler, epoch, metrics, train_loss, filename):
  torch.save({
      'epoch': epoch,
      'model_state_dict': model.state_dict(),
      'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'dice_score': metrics['dice'],
        'iou_score': metrics['iou'],
        'precision': metrics['precision'],
        'recall': metrics['recall'],
        'loss': train_loss
  },filename)

In [ ]:
def log_metrics(logger, train_loss, val_metrics, epoch):
    logger.report_scalar("Loss", "Train", train_loss, epoch)
    logger.report_scalar("Metrics", "Dice", val_metrics['dice'], epoch)
    logger.report_scalar("Metrics", "Iou", val_metrics['iou'], epoch)
    logger.report_scalar("Metrics", "Precision", val_metrics['precision'], epoch)
    logger.report_scalar("Metrics", "Recall", val_metrics['recall'], epoch)

In [ ]:
def train_model():
    setup = settings(cfg)
    device = setup['device']
    model = setup['model']
    train_loader = setup['train_loader']
    test_loader = setup['test_loader']
    criterion = setup['criterion']
    optimizer = setup['optimizer']
    scheduler = setup['scheduler']
    metrics = setup['metrics']
    task = setup['task']

    es_enabled = cfg.early_stopping.enabled
    patience = cfg.early_stopping.patience
    min_delta = cfg.early_stopping.min_delta
    restore_best = cfg.early_stopping.restore_best_weights

    best_dice = 0
    best_epoch = 0
    patience_count = 0
    best_model_weights = None

    history = {
        'train_losses': [],
        'val_dice_scores': [],
        'val_iou_scores': [],
        'val_pres_scores': [],
        'val_rec_scores': []
    }

    logger = task.get_logger()

    for epoch in range(cfg.training.epochs):
        avg_train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        val_metrics = validate_epoch(model, test_loader, metrics, device)

        if val_metrics['dice'] > best_dice:
            best_dice = val_metrics['dice']
            best_epoch = epoch
            checkpoint(
                model, optimizer, scheduler, epoch, val_metrics,
                avg_train_loss, 'best_model.pth'
            )
        log_metrics(logger, avg_train_loss, val_metrics, epoch)
        history['train_losses'].append(avg_train_loss)
        history['val_dice_scores'].append(val_metrics['dice'])
        history['val_iou_scores'].append(val_metrics['iou'])
        history['val_pres_scores'].append(val_metrics['precision'])
        history['val_rec_scores'].append(val_metrics['recall'])

        scheduler.step(avg_train_loss)

        lr = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch:03d}:')
        print(f'  Loss: {avg_train_loss:.4f}')
        print(f'  Dice: {val_metrics["dice"]:.4f}')
        print(f'  Iou: {val_metrics["iou"]:.4f}')
        print(f'  Precision: {val_metrics["precision"]:.4f}')
        print(f'  Recall: {val_metrics["recall"]:.4f}')
        print(f'  LR: {lr:.2e}')
        print(f'Best Dice {best_dice:.4f} Epoch {best_epoch}')

        should_stop, best_dice, patience_counter, best_model_weights = early_stopping(
                val_metrics['dice'],
                best_metric=best_dice,
                patience_count=patience_count,
                patience=patience,
                min_d=min_delta,
                epoch=epoch,
                model=model
        )

        if should_stop:
            print(f"Early stopping")
            if restore_best and best_model_weights is not None:
                model.load_state_dict(best_model_weights)
            break

    print(f"Best epoch: {best_epoch} Dice: {best_dice:.4f}")

    return model, history

In [ ]:
if __name__ == "__main__":
    results = train_model()

[Val]: 100%|██████████| 274/274 [01:07<00:00,  4.06it/s]


Epoch 000:
  Loss: 0.0822
  Dice: 0.9366
  Iou: 0.9125
  Precision: 0.9510
  Recall: 0.9533
  LR: 2.00e-04
Best Dice 0.9366 Epoch 0


[Val]: 100%|██████████| 274/274 [01:07<00:00,  4.08it/s]


Epoch 001:
  Loss: 0.0069
  Dice: 0.9581
  Iou: 0.9402
  Precision: 0.9668
  Recall: 0.9696
  LR: 2.00e-04
Best Dice 0.9581 Epoch 1


[Val]: 100%|██████████| 274/274 [01:07<00:00,  4.07it/s]


...

Epoch 017:
  Loss: 0.0028
  Dice: 0.9774
  Iou: 0.9671
  Precision: 0.9821
  Recall: 0.9839
  LR: 2.00e-04
Best Dice 0.9774 Epoch 17


[Val]: 100%|██████████| 274/274 [01:06<00:00,  4.10it/s]


Epoch 018:
  Loss: 0.0028
  Dice: 0.9776
  Iou: 0.9671
  Precision: 0.9814
  Recall: 0.9845
  LR: 2.00e-04
Best Dice 0.9776 Epoch 18


[Val]: 100%|██████████| 274/274 [01:06<00:00,  4.10it/s]

Epoch 019:
  Loss: 0.0027
  Dice: 0.9755
  Iou: 0.9645
  Precision: 0.9721
  Recall: 0.9915
  LR: 2.00e-04
Best Dice 0.9776 Epoch 18
Best epoch: 18 Dice: 0.9755


In [ ]:
import cv2
import os

def predict_dataset(model, dataset, device):
    model.eval()
    predictions = []
    ground_truths = []
    filenames = []
    with torch.no_grad():
        for idx in tqdm(range(len(dataset)), desc='Predicting'):
            image, mask = dataset[idx]
            filename = dataset.image_files[idx]
            image_tensor = image.unsqueeze(0).to(device)
            output = model(image_tensor)
            probs = torch.softmax(output, dim=1)
            pred_mask = torch.argmax(probs, dim=1)
            pred = pred_mask.cpu().numpy()[0]
            predictions.append(pred)
            ground_truths.append(mask.numpy())
            filenames.append(filename)

    return predictions, ground_truths, filenames

def predict_and_save(model, dataset, device, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    predictions, ground_truths, filenames = predict_dataset(model, dataset, device)
    pred_dir = os.path.join(output_dir, 'predictions')
    gt_dir = os.path.join(output_dir, 'ground_truths')
    os.makedirs(pred_dir, exist_ok=True)
    os.makedirs(gt_dir, exist_ok=True)
    for i, (pred, gt, filename) in enumerate(zip(predictions, ground_truths, filenames)):
        pred_image = (pred * 255).astype(np.uint8)
        pred_path = os.path.join(pred_dir, f"pred_{filename}")
        cv2.imwrite(pred_path, pred_image)
        gt_image = (gt * 255).astype(np.uint8)
        gt_path = os.path.join(gt_dir, f"gt_{filename}")
        cv2.imwrite(gt_path, gt_image)
    return predictions, ground_truths, filenames


In [ ]:
def evaluate_trained_model():
    checkpoint = torch.load('best_model.pth')
    setup = settings(cfg)
    model = setup['model']
    device = setup['device']
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    paths = create_split()
    _, val_transform = transformers()

    test_dataset = SegmentationDataset(
        images_dir=paths['train_images'],
        labels_dir=paths['train_labels'],
        transform=val_transform
    )
    predictions, ground_truths, filenames= predict_and_save(
        model=model,
        dataset=test_dataset,
        device=device,
        output_dir='./predictions_results',
        save_images=True
    )
    return predictions, ground_truths, filenames

In [ ]:
evaluate_trained_model()

Predicting: 100%|██████████| 21844/21844 [07:52<00:00, 46.26it/s]


([array([[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         ...,
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]]),
  array([[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         ...,
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]]),
  array([[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         ...,
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]]),
  array([[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         ...,
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]]),
  array([[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         ...,
    

In [ ]:
!zip -r my_folders.zip /kaggle/working/predictions_results/predictions

  adding: kaggle/working/predictions_results/predictions/ (stored 0%)
  adding: kaggle/working/predictions_results/predictions/pred_154_161_image.png (deflated 46%)
  adding: kaggle/working/predictions_results/predictions/pred_164_162_image.png (deflated 37%)
  adding: kaggle/working/predictions_results/predictions/pred_150_196_image.png (deflated 43%)
  adding: kaggle/working/predictions_results/predictions/pred_114_112_image.png (deflated 67%)
  adding: kaggle/working/predictions_results/predictions/pred_104_225_image.png (deflated 37%)
  adding: kaggle/working/predictions_results/predictions/pred_51_007_image.png (deflated 62%)
  adding: kaggle/working/predictions_results/predictions/pred_68_123_image.png (deflated 69%)
  adding: kaggle/working/predictions_results/predictions/pred_164_173_image.png (deflated 33%)
  adding: kaggle/working/predictions_results/predictions/pred_68_316_image.png (deflated 56%)
  adding: kaggle/working/predictions_results/predictions/pred_104_131_image.pn

In [ ]:
from IPython.display import FileLink
FileLink('my_folders.zip')

/kaggle/working/my_folders.zip